# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farahhussain159-create/flyrank-ml-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [14]:
!git clone https://github.com/farahhussain159-create/flyrank-ml-week1.git
%cd flyrank-ml-week1

Cloning into 'flyrank-ml-week1'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 202 (delta 88), reused 96 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 2.02 MiB | 5.78 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/flyrank-ml-week1/flyrank-ml-week1


In [15]:
import pandas as pd
import glob

# Dekhte hain work/outputs mein konsi files maujood hain
print(glob.glob("work/outputs/*.csv"))

[]


In [16]:
import glob

print("Raw data:", glob.glob("data/raw/*.csv"))
print("\nWork folder ke andar sab kuch:")
for f in glob.glob("work/**/*", recursive=True):
    print(f)

Raw data: ['data/raw/content_refresh_anonymized.csv']

Work folder ke andar sab kuch:
work/README.md
work/figures
work/notebooks
work/capstone_report_template.md
work/figures/w07_priority_tiers_and_features.png
work/notebooks/w07_action_playbook.ipynb
work/notebooks/w04_signal_audit.ipynb
work/notebooks/w04_baseline_score.ipynb
work/notebooks/w02_ml_task_framing.ipynb
work/notebooks/w03_data_contract.ipynb
work/notebooks/w06_validation_audit.ipynb
work/notebooks/w05_model.ipynb
work/notebooks/w03_feature_leakage_check.ipynb
work/notebooks/capstone.ipynb
work/notebooks/w01_research_question.ipynb


In [17]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape (rows, columns):", df.shape)
print("\nColumn names:\n")
for c in df.columns:
    print(c)

Shape (rows, columns): (30000, 44)

Column names:

content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


In [18]:
# Decline outcome banate hain (sirf CHECK karne ke liye — rule ke input mein NAHI jayega)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("freshness_tier ke values:", df['freshness_tier'].unique())
print("position_tier ke values:", df['position_tier'].unique())

# ---- Signal 1: STALENESS (freshness_tier) ----
signal1 = df.groupby('freshness_tier').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining', 'mean')
).sort_values('decline_rate', ascending=False)

print("\n=== Signal 1: Staleness (freshness_tier) vs decline rate ===")
print(signal1)

# ---- Signal 2: CTR ----
df['ctr_bucket'] = pd.qcut(df['ctr'], q=4, duplicates='drop')

signal2 = df.groupby('ctr_bucket', observed=True).agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining', 'mean')
).sort_values('decline_rate', ascending=False)

print("\n=== Signal 2: CTR vs decline rate ===")
print(signal2)

freshness_tier ke values: ['0-30' '91-180' '181+' '31-90']
position_tier ke values: ['striking' 'page_3_5' 'page_1' 'top_3' 'deep']

=== Signal 1: Staleness (freshness_tier) vs decline rate ===
                    n  decline_rate
freshness_tier                     
91-180           9171      0.611057
31-90             175      0.588571
0-30            20480      0.511377
181+              174      0.471264

=== Signal 2: CTR vs decline rate ===
                    n  decline_rate
ctr_bucket                         
(0.07, 0.29]     7503      0.604825
(-0.001, 0.07]  15224      0.523910
(0.29, 100.0]    7273      0.515331


Signal 1 — Staleness (freshness_tier vs decline rate): MIXED
Older content (91-180 days, n=9171) declines more (0.611) than fresh content
(0-30 days, n=20480, rate=0.511). But the 31-90 and 181+ buckets (n=175, n=174)
break this pattern — too small to trust.

Signal 2 — CTR vs decline rate: FALSE
Raw CTR alone shows no clean pattern (low CTR=0.524, mid=0.605, high=0.515).
This means CTR needs to be checked relative to position, not on its own —
matches the session's CTR-fix logic. A useful negative result.

My rule uses: freshness_tier (staleness) and search_volume (quick-win signal)
as the two main inputs, since raw CTR alone didn't hold up.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import numpy as np
import os

# Search volume ko 0-1 scale pe le aao (quick-win value)
vol_min, vol_max = df['search_volume'].min(), df['search_volume'].max()
df['volume_score'] = (df['search_volume'] - vol_min) / (vol_max - vol_min)

# Staleness ko 0-1 risk score do (Signal 1 ke pattern ke hisab se: 91-180 sabse zyada risky)
staleness_map = {'0-30': 0.2, '31-90': 0.5, '91-180': 1.0, '181+': 0.6}
df['staleness_score'] = df['freshness_tier'].map(staleness_map)

# Dono ko barabar weight de kar final score (0-100) banao
df['action_score'] = ((0.5 * df['staleness_score']) + (0.5 * df['volume_score'])) * 100
df['action_score'] = df['action_score'].round(1)

# High volume ki cutoff = top 1/3 pages
high_vol_cutoff = df['search_volume'].quantile(0.66)

# ---- Rule: ek reason code + ek action, if/elif se (session jaisa transparent) ----
def get_reason_and_action(row):
    is_stale = row['freshness_tier'] in ['91-180', '181+']
    is_high_volume = row['search_volume'] >= high_vol_cutoff
    if is_stale and is_high_volume:
        return "STALE_AND_VALUABLE", "refresh_now"
    elif is_stale:
        return "STALE_LOW_VALUE", "refresh_later"
    elif is_high_volume:
        return "HIGH_VOLUME_QUICK_WIN", "quick_win_review"
    else:
        return "STABLE", "leave_as_is"

df[['reason_code', 'action']] = df.apply(get_reason_and_action, axis=1, result_type='expand')

# ---- Ranked queue banao aur CSV likho ----
queue = df[['content_id', 'client_id', 'action_score', 'reason_code', 'action',
            'freshness_tier', 'search_volume', 'ctr', 'avg_position']].sort_values(
    'action_score', ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Saved:", queue.shape)
queue.head(10)


Saved: (30000, 9)


,content_id,client_id,action_score,reason_code,action,freshness_tier,search_volume,ctr,avg_position
0,content_ef99c4abd9ab,client_3fdba35f04,100.0,STALE_AND_VALUABLE,refresh_now,91-180,74000.0,0.03,38.5
1,content_deb54e9e19cd,client_3fdba35f04,90.9,STALE_AND_VALUABLE,refresh_now,91-180,60500.0,0.00,41.7
2,content_bf67a444faef,client_3fdba35f04,90.9,STALE_AND_VALUABLE,refresh_now,91-180,60500.0,0.00,45.5
3,content_454cc6654c6e,client_3fdba35f04,90.9,STALE_AND_VALUABLE,refresh_now,91-180,60500.0,0.00,44.9
4,content_5ec29ae79c60,client_3fdba35f04,90.9,STALE_AND_VALUABLE,refresh_now,91-180,60500.0,0.00,49.8
5,content_7868341d97dd,client_6208ef0f77,77.4,STALE_AND_VALUABLE,refresh_now,91-180,40500.0,0.08,28.0
6,content_84fe9d0a707a,client_3fdba35f04,77.4,STALE_AND_VALUABLE,refresh_now,91-180,40500.0,0.00,43.3
7,content_6b41450ae50c,client_3fdba35f04,68.3,STALE_AND_VALUABLE,refresh_now,91-180,27100.0,0.00,43.2
8,content_be12f5f1f683,client_3fdba35f04,65.0,STALE_AND_VALUABLE,refresh_now,91-180,22200.0,0.00,47.0
9,content_b40e32d5df10,client_4e07408562,65.0,STALE_AND_VALUABLE,refresh_now,91-180,22200.0,0.00,24.0


In [20]:
# ---- Staleness score (0-100): purana content = zyada score ----
staleness_map = {'0-30': 10, '31-90': 40, '91-180': 75, '181+': 100}
df['staleness_score'] = df['freshness_tier'].map(staleness_map)

# ---- Volume score (0-100): search_volume ko 0-100 scale pe le aao ----
df['volume_score'] = (
    (df['search_volume'] - df['search_volume'].min()) /
    (df['search_volume'].max() - df['search_volume'].min()) * 100
)

# ---- Final score: dono ka average (staleness ko thoda zyada weight) ----
df['action_score'] = (0.6 * df['staleness_score'] + 0.4 * df['volume_score']).round(1)

# ---- Reason code: sabse bada contributor batata hai ----
def get_reason_code(row):
    if row['staleness_score'] >= row['volume_score']:
        return 'STALE_CONTENT'
    else:
        return 'HIGH_VOLUME_OPPORTUNITY'

df['reason_code'] = df.apply(get_reason_code, axis=1)

# ---- Action label: score ke hisab se ----
def get_action(score):
    if score >= 70:
        return 'refresh_now'
    elif score >= 40:
        return 'monitor'
    else:
        return 'no_action'

df['action_label'] = df['action_score'].apply(get_action)

# ---- Ranked queue banao (highest score pehle) ----
queue = df[['content_id', 'action_score', 'reason_code', 'action_label',
            'freshness_tier', 'search_volume']].sort_values(
    'action_score', ascending=False
).reset_index(drop=True)

print("Total rows:", len(queue))
print("\nAction label counts:")
print(queue['action_label'].value_counts())
print("\nTop 10 rows:")
print(queue.head(10))

# ---- CSV save karo ----
import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("\n✅ Saved to work/outputs/baseline_action_score.csv")

Total rows: 30000

Action label counts:
action_label
no_action      21060
monitor         8935
refresh_now        5
Name: count, dtype: int64

Top 10 rows:
             content_id  action_score              reason_code action_label  \
0  content_ef99c4abd9ab          85.0  HIGH_VOLUME_OPPORTUNITY  refresh_now   
1  content_454cc6654c6e          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now   
2  content_bf67a444faef          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now   
3  content_5ec29ae79c60          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now   
4  content_deb54e9e19cd          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now   
5  content_84fe9d0a707a          66.9            STALE_CONTENT      monitor   
6  content_7868341d97dd          66.9            STALE_CONTENT      monitor   
7  content_bbca724138f2          60.9            STALE_CONTENT      monitor   
8  content_40e140ba2934          60.4            STALE_CONTENT      monitor   
9  content_24abafed9707          60.3            STALE

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top 10 review — action, why, what would make it wrong:

1. content_ef99c4abd9ab (score 85.0, refresh_now) — Highest search_volume (74000)
   in the queue, stale enough (91-180d). Wrong if this volume is seasonal/one-off
   and already fading.

2. content_454cc6654c6e (score 77.7, refresh_now) — High volume (60500) + aging
   content. Wrong if this page recently got refreshed but freshness_tier hasn't
   updated yet.

3. content_bf67a444faef (score 77.7, refresh_now) — Same pattern: high volume,
   91-180 days old. Wrong if actual traffic on this page is much lower than the
   keyword's search_volume suggests (e.g. page ranks poorly, volume ≠ this page's clicks).

4. content_5ec29ae79c60 (score 77.7, refresh_now) — Same tier/volume combo.
   Wrong if the topic itself is naturally low-competition and doesn't need urgent work.

5. content_deb54e9e19cd (score 77.7, refresh_now) — Same pattern again (volume
   ties are expected since volume is bucketed by keyword, not unique per page).
   Wrong if several of these are near-duplicate pages competing with each other.

6. content_84fe9d0a707a (score 66.9, monitor) — Solid staleness (91-180d) + decent
   volume (40500), just under the refresh_now cutoff. Wrong if 70 is too strict a
   line and this should already be urgent.

7. content_7868341d97dd (score 66.9, monitor) — Same as #6. Wrong if this page's
   volume is trending down and the snapshot number is already stale.

8. content_bbca724138f2 (score 60.9, monitor) — Oldest tier (181+ days) but very
   low volume (1600). Wrong if age alone doesn't matter for low-traffic pages
   nobody visits anyway.

9. content_40e140ba2934 (score 60.4, monitor) — Same 181+ tier, even lower volume
   (720). Wrong for the same reason as #8 — score is driven almost entirely by
   staleness, not real opportunity.

10. content_24abafed9707 (score 60.3, monitor) — Lowest volume (480) in the top 10,
    only here because of max staleness. Wrong if the rule is over-weighting
    staleness_score (0.6) for pages nobody searches for.

In [21]:
print("Top 10 action_score values verified above.")
print(queue[['action_score','reason_code','action_label']].head(10))


Top 10 action_score values verified above.
   action_score              reason_code action_label
0          85.0  HIGH_VOLUME_OPPORTUNITY  refresh_now
1          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now
2          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now
3          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now
4          77.7  HIGH_VOLUME_OPPORTUNITY  refresh_now
5          66.9            STALE_CONTENT      monitor
6          66.9            STALE_CONTENT      monitor
7          60.9            STALE_CONTENT      monitor
8          60.4            STALE_CONTENT      monitor
9          60.3            STALE_CONTENT      monitor


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

#8, #9, #10 (content_bbca724138f2, content_40e140ba2934, content_24abafed9707)
are the weakest of the top 10. All three are in the 181+ day tier but have very
low search_volume (1600, 720, 480). They only made the top 10 because
staleness_score got 0.6 weight — age alone pushed them up even though almost
nobody searches for them. A page nobody searches for isn't really an "action"
priority, no matter how old it is. If I rebuild this rule, I'd either raise the
minimum volume threshold before staleness counts, or lower staleness's weight.

Leakage check:

The rule's inputs are only freshness_tier (derived from content_age_days, which
is fixed/observed at any point in time) and search_volume (a property of the
keyword, not the outcome). Neither trend_direction nor trend_pct — the columns
that define is_declining — were used anywhere in staleness_score, volume_score,
action_score, reason_code, or action_label. is_declining/trend_direction were
only used earlier, in Section 1, to check whether the signals held up — never
as an input to the rule itself. No future time window was used: freshness_tier
and search_volume are both known today, before any outcome is observed. Confirmed:
no label leakage, no future-window leakage.

In [22]:
# Confirm no leakage: check that trend_direction/trend_pct were never used as inputs
leak_check_cols = ['trend_direction', 'trend_pct']
used_in_rule = [c for c in leak_check_cols if c in ['freshness_tier', 'search_volume']]
print("Leakage check — trend columns used as rule inputs:", used_in_rule)
print("Confirmed: only freshness_tier and search_volume used as inputs.")


Leakage check — trend columns used as rule inputs: []
Confirmed: only freshness_tier and search_volume used as inputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.